In [ ]:
# ================================================================
# 0. IMPORTAR LIBRERÍAS
# ================================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
import joblib
from xgboost import XGBClassifier

# ================================================================
# 1. MAPEO DE TARGET
# ================================================================
mapa_target = {
    "bajo": 0,
    "medio-bajo": 1,
    "medio-alto": 2,
    "alto": 3
}
mapa_inverso = {v: k for k, v in mapa_target.items()}

# ================================================================
# 2. FUNCIÓN DE PREPROCESAMIENTO COMPLETA
# ================================================================
def preprocesar_pipeline(df, y_col=None, id_col_name="ID",
                         generar_csv_individual=True, train=True, prefijo_csv="train"):

    df = df.copy()

    # ----------------------------
    # Separar ID y target
    # ----------------------------
    id_col = df[id_col_name].reset_index(drop=True)

    if y_col is not None:
        y = df[y_col].reset_index(drop=True)

        y = y.map(mapa_target)
        if y.isnull().any():
            raise ValueError("❌ ERROR: El target contiene valores no válidos.")
        df = df.drop(columns=[y_col])
    else:
        y = None

    # ----------------------------
    # Eliminar columnas innecesarias
    # ----------------------------
    cols_a_eliminar = [
        "E_VALORMATRICULAUNIVERSIDAD",
        "F_TIENELAVADORA",
        "INDICADOR_1",
        "INDICADOR_2",
        "INDICADOR_3",
        "INDICADOR_4",
        "PERIODO_ACADEMICO"
    ]
    df = df.drop(columns=[c for c in cols_a_eliminar if c in df.columns])

    # ----------------------------
    # Identificar tipos de columnas
    # ----------------------------
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
    cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

    if id_col_name in cat_cols:
        cat_cols.remove(id_col_name)

    # =============================================================
    #  PROCESAMIENTO NUMÉRICO
    # =============================================================
    if train:
        imp_num = SimpleImputer(strategy="median")
        scaler = RobustScaler()

        X_num = imp_num.fit_transform(df[num_cols])
        X_num = scaler.fit_transform(X_num)

        joblib.dump(imp_num, "imp_num.pkl")
        joblib.dump(scaler, "scaler.pkl")

    else:
        imp_num = joblib.load("imp_num.pkl")
        scaler = joblib.load("scaler.pkl")

        X_num = imp_num.transform(df[num_cols])
        X_num = scaler.transform(X_num)

    X_num = pd.DataFrame(X_num, columns=num_cols).reset_index(drop=True)

    # =============================================================
    #  PROCESAMIENTO CATEGÓRICO
    # =============================================================
    if train:
        imp_cat = SimpleImputer(strategy="most_frequent")
        ohe_dict = {}
        freq_dict = {}
        joblib.dump(imp_cat, "imp_cat.pkl")
    else:
        imp_cat = joblib.load("imp_cat.pkl")
        ohe_dict = joblib.load("ohe_dict.pkl")
        freq_dict = joblib.load("freq_dict.pkl")

    all_cat_processed = []

    for col in cat_cols:
        col_data = imp_cat.fit_transform(df[[col]]) if train else imp_cat.transform(df[[col]])
        col_series = pd.Series(col_data.ravel(), name=col).reset_index(drop=True)

        # Determinar si OHE o frecuencia
        max_unique = 10

        if train:
            if col_series.nunique() <= max_unique:
                ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
                col_ohe = ohe.fit_transform(col_series.values.reshape(-1, 1))
                ohe_dict[col] = ohe
                df_col = pd.DataFrame(col_ohe, columns=ohe.get_feature_names_out([col]))
            else:
                freq_map = col_series.value_counts(normalize=True)
                df_col = col_series.map(freq_map).fillna(0).to_frame(f"{col}_freq")
                freq_dict[col] = freq_map

        else:
            if col in ohe_dict:  # OHE
                ohe = ohe_dict[col]
                col_ohe = ohe.transform(col_series.values.reshape(-1, 1))
                df_col = pd.DataFrame(col_ohe, columns=ohe.get_feature_names_out([col]))
            else:  # Frecuencias
                freq_map = freq_dict[col]
                df_col = col_series.map(freq_map).fillna(0).to_frame(f"{col}_freq")

        df_col = df_col.reset_index(drop=True)
        all_cat_processed.append(df_col)

    if train:
        joblib.dump(ohe_dict, "ohe_dict.pkl")
        joblib.dump(freq_dict, "freq_dict.pkl")

    # =============================================================
    # UNIR DATAFRAME COMPLETO
    # =============================================================
    df_completo = pd.concat([X_num] + all_cat_processed, axis=1)

    df_completo.insert(0, "ID", id_col)

    if y is not None:
        df_completo["RENDIMIENTO_GLOBAL"] = y

    df_completo.to_csv(f"{prefijo_csv}_preprocesado_completo.csv", index=False)

    return df_completo

# ================================================================
# 3. CARGAR Y PREPROCESAR TRAIN
# ================================================================
df_train = pd.read_csv("/kaggle/input/ejercicio/train.csv")

df_train_proc = preprocesar_pipeline(
    df_train,
    y_col="RENDIMIENTO_GLOBAL",
    id_col_name="ID",
    generar_csv_individual=False,
    train=True,
    prefijo_csv="train"
)

X = df_train_proc.drop(columns=["ID", "RENDIMIENTO_GLOBAL"])
y = df_train_proc["RENDIMIENTO_GLOBAL"]

# ================================================================
# 4. VALIDACIÓN CRUZADA
# ================================================================
xgb_cv = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softmax",
    num_class=4,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

cv_scores = cross_val_score(xgb_cv, X, y, cv=5, scoring="accuracy")
print(f"Accuracy CV 5-fold: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# ================================================================
# 5. ENTRENAR MODELO FINAL
# ================================================================
xgb_cv.fit(X, y)
joblib.dump(xgb_cv, "xgb_model_final.pkl")

print("✅ Modelo XGBoost guardado")

# ================================================================
# 6. CARGAR TEST Y PREPROCESAR
# ================================================================
df_test = pd.read_csv("/kaggle/input/ejercicio/test.csv")

df_test_proc = preprocesar_pipeline(
    df_test,
    y_col=None,
    id_col_name="ID",
    generar_csv_individual=False,
    train=False,
    prefijo_csv="test"
)

# ================================================================
# 7. AJUSTAR COLUMNAS
# ================================================================
X_test = df_test_proc.drop(columns=["ID"])

missing_cols = set(X.columns) - set(X_test.columns)
for col in missing_cols:
    X_test[col] = 0

extra_cols = set(X_test.columns) - set(X.columns)
if extra_cols:
    X_test.drop(columns=list(extra_cols), inplace=True)

X_test = X_test[X.columns]

# ================================================================
# 8. PREDICCIÓN Y SUBMISSION
# ================================================================
pred_test = xgb_cv.predict(X_test)
pred_labels = [mapa_inverso[p] for p in pred_test]

submission = pd.DataFrame({
    "ID": df_test_proc["ID"],
    "RENDIMIENTO_GLOBAL": pred_labels
})

submission.to_csv("submission2.csv", index=False)
print("✅ Submission generado")
print(submission.head())
